# Game EDA

Scoring, margins, home/away outcomes, scoring seasonality and overtime where source fields permit.

In [ ]:
from pathlib import Path
import sys

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / 'backend' / 'app').is_dir())
sys.path.insert(0, str(ROOT)) if str(ROOT) not in sys.path else None
import pandas as pd
import plotly.express as px
from backend.app import eda

DATA_DIRS = [ROOT / 'data' / 'processed', ROOT / 'data' / 'raw', ROOT / 'Raw']
games = eda.games_analysis(DATA_DIRS)
if games['available']:
    display({key: games.get(key) for key in ('game_count', 'average_points_per_team', 'average_winning_margin', 'home_win_percentage', 'overtime', 'unavailable_analyses')})
else:
    print(games['reason'])

In [ ]:
if games.get('scoring_by_season'):
    scoring = pd.DataFrame(games['scoring_by_season'])
    y = 'average_points_per_team' if 'average_points_per_team' in scoring else 'points'
    px.line(scoring, x='season', y=y, markers=True, title='Average points per team by season').show()
else:
    print('Season scoring trend unavailable.')
display(pd.DataFrame(games.get('highest_scoring_games', [])))
display(pd.DataFrame(games.get('lowest_scoring_games', [])))

In [ ]:
distribution = games.get('score_distribution', {})
if distribution.get('bin_centers'):
    px.bar(x=distribution['bin_centers'], y=distribution['counts'],
           labels={'x': 'Game total points (bin center)', 'y': 'Games'},
           title='Game scoring distribution').show()
box = games.get('score_boxplot', {})
if box.get('home_points') and box.get('away_points'):
    box_data = pd.DataFrame({'points': box['home_points'] + box['away_points'],
                             'venue': ['Home'] * len(box['home_points']) + ['Away'] * len(box['away_points'])})
    px.box(box_data, x='venue', y='points', title='Home vs away team points (deterministic sample)').show()
display(games.get('home_away_win_distribution', {}))